<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/chat_template_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Chat Templating

Doc being followed: https://huggingface.co/learn/llm-course/chapter11/2

## Installs


In [14]:
!pip install dataset
!pip install transformers

## Import dataset

In [10]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

data/all/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  223MB            

data/all/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/all/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  223MB            

data/all/train-00001-of-00009.parquet: downloading bytes:           |  0.00B            

data/all/train-00002-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  223MB            

...

data/all/train-00008-of-00009.parquet: downloading bytes:           |  0.00B            

data/all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  105MB            

data/all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 1043917
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 54948
    })
})

## Get Tokenizers

We will be taking a look at Mistrals, Qwens and Smol Chats tokenizer

In [50]:
from transformers import AutoTokenizer

# No suppport for tool calling - simple instruct models
mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct")
smol_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

# Support for tool calling - more advanced instruct models
qwen25_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")


[transformers] loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.1/snapshots/ec5deb64f2c6e6fa90c1abf74a91d5c93a9669ca/config.json
[transformers] Model config MistralConfig {
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 32768,
  "model_type": "mistral",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "sliding_window": 4096,
  "tie_word_embeddings": false,
  "transformers_version": "5.17.0",
  "use_cache": true,
  "vocab_size": 32000
}

[transformers] loading configuration file config.json from cache 

## Play with Tokenizers

Try out different different sets of messages

In [32]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
]

# Using smol_tokenizer as it has a default chat_template configured
print(smol_tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>system
You are a helpful assistant.<|im_end|>



In [31]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]

print(smol_tokenizer.apply_chat_template(messages, tokenize=False))
print(qwen_tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>



Trying a tokenizer with tool support

In [51]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate math operations",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string", "enum": ["multiply"]},
                    "x": {"type": "integer"},
                    "y": {"type": "integer"}
                },
                "required": ["operation", "x", "y"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "weather_api",
            "description": "Get current weather details",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "country": {"type": "string"}
                },
                "required": ["city", "country"]
            }
        }
    }
]

messages = [
    {
        "role": "system",
        "content": "You are an AI assistant that can use tools."
    },
    {
        "role": "user",
        "content": "What's 123 * 456 and is it raining in Paris?"
    },
    {
        "role": "assistant",
        "content": "Let me help you with that.",
        "tool_calls": [
            {
                "type": "function",
                "function": {
                    "name": "calculator",
                    "arguments": '{"operation": "multiply", "x": 123, "y": 456}'
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "weather_api",
                    "arguments": '{"city": "Paris", "country": "France"}'
                }
            }
        ]
    },
    {
        "role": "tool",
        "name": "calculator",
        "content": "56088"
    },
    {
        "role": "tool",
        "name": "weather_api",
        "content": "{\"condition\": \"rain\", \"temperature\": 15}"
    }
]

In [54]:
templated_prompt_qwen25 = qwen25_tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

print(templated_prompt_qwen25)

<|im_start|>system
You are an AI assistant that can use tools.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "calculator", "description": "Calculate math operations", "parameters": {"type": "object", "properties": {"operation": {"type": "string", "enum": ["multiply"]}, "x": {"type": "integer"}, "y": {"type": "integer"}}, "required": ["operation", "x", "y"]}}}
{"type": "function", "function": {"name": "weather_api", "description": "Get current weather details", "parameters": {"type": "object", "properties": {"city": {"type": "string"}, "country": {"type": "string"}}, "required": ["city", "country"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>u